# Cleaning CSV

### installing pandas and kagglehub


In [3]:
%pip install pandas
%pip install kagglehub
%pip install scipy
%pip install numpy
%pip install seaborn
%pip install matplotlib


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgr

### Importing pandas library and the dataset

In [ ]:
import pandas as pd
print(pd.__version__)

import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "unclean_customer_data.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "mohammedarfathr/customer-transactions-dataset",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())



3.0.6


/var/folders/wx/srl0047n7ggf6r_b5glkrctr0000gn/T/ipykernel_68252/2670071190.py:11: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


First 5 records:    Customer_ID   Age  Annual_Income  Spending_Score  Purchase_Frequency  \
0         8270  31.0   75851.592212       77.104268                  17   
1         1860  48.0   44460.183764       61.349076                  25   
2         6390  17.0   43677.755605       22.589956                   9   
3         6191  34.0   48666.812544       82.069525                   0   
4         6734  44.0   46080.334029       65.076915                   3   

   Transaction_Amount  
0          129.724190  
1           91.479265  
2          130.573812  
3           72.395846  
4          156.399077  


### PreAnalysis 1: Cleaning column names

Firstly i am cleaning the column names in the data. im using .strip() to remove whitespace at the start and end of the column names, .lower() to make them all lowercase, and .replace() to replace whitespace with undescores (_).

In [67]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

## Exploratory Data Analysis
### PreAnalysis 2: Inspecting the data
Data contains:

customer id: unique id num per customer \
age: age of the customer \
annual income: annual income in USD \
spending score: a score between 1 and 100 representing spending habits \
purchase frequency: number of purchases made in a month \
transaction amount: average transaction amount in USD

In [68]:
df.head(5)

,customer_id,age,annual_income,spending_score,purchase_frequency,transaction_amount
0,8270,31.0,75851.592212,77.104268,17,129.724190
1,1860,48.0,44460.183764,61.349076,25,91.479265
2,6390,17.0,43677.755605,22.589956,9,130.573812
3,6191,34.0,48666.812544,82.069525,0,72.395846
4,6734,44.0,46080.334029,65.076915,3,156.399077


In [69]:
# print the dimensions of the dataframe
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[0]:,}")

# print first 10 rows
df.head(10)

# inspect the columns and datatypes for each
df.info()

# summary
display(df.describe(include="all").T)

Rows: 10,200
Columns: 10,200
<class 'pandas.DataFrame'>
RangeIndex: 10200 entries, 0 to 10199
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   customer_id         10200 non-null  int64  
 1   age                 9943 non-null   float64
 2   annual_income       10200 non-null  float64
 3   spending_score      10200 non-null  float64
 4   purchase_frequency  10200 non-null  int64  
 5   transaction_amount  9944 non-null   float64
dtypes: float64(4), int64(2)
memory usage: 478.3 KB


,count,mean,std,min,25%,50%,75%,max
customer_id,10200.0,5466.655392,2591.699853,1001.000000,3211.750000,5485.000000,7708.250000,9998.000000
age,9943.0,34.458212,9.998262,0.000000,28.000000,34.000000,41.000000,69.000000
annual_income,10200.0,50124.610200,14915.806974,-17177.834332,40002.275881,50091.128413,60290.094248,112218.425755
spending_score,10200.0,50.587613,28.628536,1.009234,26.110644,50.287120,75.637159,99.990069
purchase_frequency,10200.0,14.507255,8.662804,0.000000,7.000000,15.000000,22.000000,29.000000
transaction_amount,9944.0,100.190612,49.846288,-86.593231,66.144312,100.177920,133.994751,273.518342


From the above initial info we can see that Age, and Transaction amount are missing some values.

Min and Max for [Age, Spending_Score, Purchase_Frequency] are reasonable ranges.

Transaction_Amount contains negative values, this needs to be investigated.

### This shows 2 initial problems: remove noise in Transaction_Amount, and investigate missing values in Age and Transaction_amount

## Step 1: identify Missing Values

In [71]:
missing_values_df = pd.DataFrame({
    "Column": ["age", "transaction_amount", "customer_id", "annual_income", "spending_score", "purchase_frequency"],
    "num missing": [df["age"].isna().sum(),
    df["transaction_amount"].isna().sum(),
    df["customer_id"].isna().sum(),
    df["annual_income"].isna().sum(),
    df["spending_score"].isna().sum(),
    df["purchase_frequency"].isna().sum(),
    ]
})

display(missing_values_df)

,Column,num missing
0,age,257
1,transaction_amount,256
2,customer_id,0
3,annual_income,0
4,spending_score,0
5,purchase_frequency,0


## Step 2: detect outliers in numerical columns

In [ ]:
#imports
import numpy as np
from scipy.stats import zscore

# firstly lets look at only the columns which are numeric
numeric_cols = df.select_dtypes(include="number").columns.difference(["customer_id"])

# next lets check for statistical outliers using Z scoreS
# new dataframe for only numeric columns and showing outliers
outliers_df = pd.DataFrame({
    "Column": numeric_cols,
    #"Z Score": z_score,
    "Outliers": [
        (np.abs(zscore(df[col].dropna())) > 3).sum()
        for col in numeric_cols,
    ]
    "Skew": [df[col].skew
    for col in numeric_cols,
    ]
})

display(outliers_df)

SyntaxError: invalid syntax (2569025531.py, line 15)

From using Z scores to find outliers we can see that annual_income and transaction_amount have some statistical outliers.

### we need to clean the following:
missing values \
outliers\
skewed \
unnormalised \
extreme values \
noise